In [10]:
# =========================
# IMPORTS
# =========================
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib
import gradio as gr
from transformers import pipeline



# =========================
# HELPERS
# =========================
def normalize_links(text: str) -> str:
    """Replace URLs with <URL> placeholder"""
    return re.sub(r'http\S+|www\S+', "<URL>", text)

# =========================
# LOAD TRANSLATOR (any language → English)
# =========================
translation_model_name = "Helsinki-NLP/opus-mt-mul-en"
translator = pipeline("translation", model=translation_model_name)

def translate_to_english(text):
    try:
        normalized = normalize_links(text)
        result = translator(normalized, max_length=512)
        return result[0]['translation_text']
    except Exception:
        return text  # fallback

# =========================
# LOAD HUGGING FACE SPAM MODEL (choose one)
# =========================
# =========================
# LOAD HUGGING FACE SPAM MODEL (new one)
# =========================
translation_model_name = "Helsinki-NLP/opus-mt-mul-en"
translator = pipeline("translation", model=translation_model_name)

hf_pipe = pipeline("text-classification", model="mrm8488/bert-tiny-finetuned-sms-spam-detection")

## =========================
# LOCAL DATASET (load from Google Drive)
# =========================
import os
from google.colab import files, drive
import shutil

drive.mount('/content/drive')

dataset_path = "/content/drive/MyDrive/spam.csv"

if not os.path.exists(dataset_path):
    print("⚠️ spam.csv not found in Drive. Please upload it now.")
    uploaded = files.upload()   # Choose spam.csv from your computer
    shutil.move(list(uploaded.keys())[0], dataset_path)
    print(f"✅ File saved permanently to {dataset_path}")

# Now load dataset safely
data = pd.read_csv(dataset_path, encoding='latin-1', on_bad_lines='skip')
data = data.rename(columns={'v1': 'label', 'v2': 'text'})[['label', 'text']]
data['label'] = data['label'].map({'ham': 0, 'spam': 1})

# 🔑 Normalize links in training
X = data['text'].apply(normalize_links)
y = data['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train local model
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)

# Evaluate local model
y_pred = clf.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Not Spam', 'Spam']))

# =========================
# PREDICTION HELPERS
# =========================
def predict_sms_local(text):
    english_text = translate_to_english(text)
    text_tfidf = vectorizer.transform([english_text])
    prob = clf.predict_proba(text_tfidf)[0]
    pred = clf.predict(text_tfidf)[0]
    label = "SPAM" if pred == 1 else "HAM"
    confidence = prob[1] * 100 if pred == 1 else prob[0] * 100
    return label, confidence
def predict_sms_hf(text):
    text_norm = normalize_links(text)
    result = hf_pipe(text_norm)[0]
    label = "SPAM" if result['label'].lower() == "spam" else "HAM"
    confidence = result['score'] * 100
    return label, confidence

# =========================
# ENSEMBLE PREDICTION
# =========================
HIGH_CONF_THRESHOLD = 85.0

def predict_sms_ensemble(text):
    text_norm = normalize_links(text)
    local_label, local_conf = predict_sms_local(text_norm)
    hf_label, hf_conf = predict_sms_hf(text_norm)

    # Special rule for messages with links
    if "<URL>" in text_norm:
        if local_label == "SPAM" and hf_label == "SPAM":
            return (f"🚨 Spam — Both models agree (contains link)\n\n"
                    f"Local Model: {local_label} ({local_conf:.2f}% confidence)\n"
                    f"HF Model: {hf_label} ({hf_conf:.2f}% confidence)")
        else:
            return (f"⚠️ Suspicious — Link detected, but models disagree\n\n"
                    f"Local Model: {local_label} ({local_conf:.2f}% confidence)\n"
                    f"HF Model: {hf_label} ({hf_conf:.2f}% confidence)")

    # Normal (non-link) cases
    if local_label == "HAM" and local_conf >= HIGH_CONF_THRESHOLD:
        final_decision = "✅ Not Spam — Local Model High Confidence"
    elif local_label == "SPAM" and hf_label == "SPAM":
        final_decision = "🚨 Spam — Both Models Agree"
    else:
        final_decision = "⚠️ Suspicious — Models Disagree"

    return (f"{final_decision}\n\n"
            f"Local Model: {local_label} ({local_conf:.2f}% confidence)\n"
            f"HF Model: {hf_label} ({hf_conf:.2f}% confidence)")
# =========================
# GRADIO UI
# =========================
with gr.Blocks() as demo:
    gr.Markdown("## 🌍 Multilingual Fraud / Spam SMS Detector")

    sms_input = gr.Textbox(label="Enter SMS text", placeholder="Type your message here... (any language)")
    with gr.Row():
        predict_btn = gr.Button("Predict (Ensemble)")

    output = gr.Textbox(label="Result")

    predict_btn.click(fn=predict_sms_ensemble, inputs=sms_input, outputs=output)

demo.launch()

import joblib

# Save model
joblib.dump(clf, "spam_model.pkl")

# Save vectorizer
joblib.dump(vectorizer, "vectorizer.pkl")



/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu
Device set to use cpu
Device set to use cpu


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Accuracy: 0.9668161434977578
Confusion Matrix:
 [[942   0]
 [ 37 136]]
              precision    recall  f1-score   support

    Not Spam       0.96      1.00      0.98       942
        Spam       1.00      0.79      0.88       173

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://910fd9a1b30a19a745.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the termin

['vectorizer.pkl']

In [ ]:
from google.colab import drive
drive.mount('/content/drive')